In [1]:
!pip3 install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 90.5 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [38]:
# get the the secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

wandb.login(key=WANDB_API_KEY)
login(token=HF_TOKEN)

print ("Authentication works with WANDB and HF")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Authentication works with WANDB and HF


In [39]:
RUN_VERSION = "v1" 
model_name  = "distilbert-base-uncased"
CONFIGS = {
    "v1": {"learning_rate": 3e-5, "batch_size": 16, "epochs": 3},
    "v2": {"learning_rate": 5e-5, "batch_size": 32, "epochs": 3},
}
cfg = CONFIGS[RUN_VERSION]
MAX_LENGTH = 512
print(f"Running version : {RUN_VERSION}")
print(f"Learning rate   : {cfg['learning_rate']}")
print(f"Batch size      : {cfg['batch_size']}")
print(f"Epochs          : {cfg['epochs']}")

Running version : v1
Learning rate   : 3e-05
Batch size      : 16
Epochs          : 3


In [40]:
# Initialise WANDB

import wandb
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
 
# 1. Initialise W&B
wandb.init(
    project="mlops-group-project",
    name="distilbert-run-v1",
	config={
    	"model": model_name,
        "dataset": "dair-ai/emotion",
        "epochs": cfg["epochs"],
        "batch_size": cfg["batch_size"],
        "learning_rate": cfg['learning_rate'],
        "max_length": MAX_LENGTH,
        "dataset": "UCSD Goodreads",
        "platform": "Kaggle",
        "version": RUN_VERSION
	}
)
 

eval/accuracy,▁█▄
eval/f1,▄█▁
eval/loss,█▁▆
eval/runtime,█▂▁
eval/samples_per_second,▁▇█
eval/steps_per_second,▁▇█
train/epoch,▁▁▁▂▂▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
train/global_step,▁▁▁▂▂▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
train/grad_norm,▂▄▄▂▂▃▃▂▆▂▇▃▃▃▅▁▂▂▃▂▂▂▂█▅▂▂▂▁▂▄▄▂▂▃
train/learning_rate,▄███▇▄███▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁
+1,...


In [41]:
# Load the dataset
from datasets import load_dataset
from collections import Counter

raw = load_dataset("dair-ai/emotion")

In [42]:
# print the dataset and labels of emotions
print(raw)

label_names = raw["train"].features["label"].names
print("\nLabel names from dataset:", label_names)

# Build id2label from the dataset itself
id2label = {i: name for i, name in enumerate(label_names)}
print("id2label:", id2label)

print("\nClass distribution (train):")
print(Counter(raw["train"]["label"]))

print("\nSample:")
for i in range(3):
    label_int = raw["train"][i]["label"]
    print(f"  '{raw['train'][i]['text']}'  →  {label_int} ({id2label[label_int]})")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Label names from dataset: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
id2label: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

Class distribution (train):
Counter({1: 5362, 0: 4666, 3: 2159, 4: 1937, 2: 1304, 5: 572})

Sample:
  'i didnt feel humiliated'  →  0 (sadness)
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'  →  0 (sadness)
  'im grabbing a minute to post i feel greedy wrong'  →  3 (anger)


In [48]:
# Cleaning the dataset 

# Lowercase  — DistilBERT-uncased expects lowercase input
# Strip URLs — carry no emotional signal
# Strip @mentions & #hashtags 
# Collapse whitespace — normalises newline/multi-space artefacts

import re

def clean_text(example):
    text = example["text"].lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return {"text": text}

cleaned = raw.map(clean_text)

# Remove any rows where text became empty after cleaning
cleaned = cleaned.filter(lambda x: len(x["text"]) > 0)

print(f"Train after cleaning: {len(cleaned['train'])} samples")
print(f"Test  after cleaning: {len(cleaned['test'])}  samples")

Train after cleaning: 16000 samples
Test  after cleaning: 2000  samples


In [49]:
# print the dataset and labels of emotions
print(cleaned)

cleaned_label_names = cleaned["train"].features["label"].names
print("\nLabel names from dataset:", cleaned_label_names)

# Build id2label from the dataset itself
cleaned_id2label = {i: name for i, name in enumerate(cleaned_label_names)}
print("id2label:", cleaned_id2label)

print("\nClass distribution (train):")
print(Counter(cleaned["train"]["label"]))

print("\nSample:")
for i in range(3):
    label_int = cleaned["train"][i]["label"]
    print(f"  '{cleaned['train'][i]['text']}'  →  {label_int} ({cleaned_id2label[label_int]})")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Label names from dataset: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
id2label: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

Class distribution (train):
Counter({1: 5362, 0: 4666, 3: 2159, 4: 1937, 2: 1304, 5: 572})

Sample:
  'i didnt feel humiliated'  →  0 (sadness)
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'  →  0 (sadness)
  'im grabbing a minute to post i feel greedy wrong'  →  3 (anger)


In [50]:
# Tokenisation 
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

train_texts  = list(cleaned["train"]["text"])
train_labels = list(cleaned["train"]["label"])
test_texts   = list(cleaned["test"]["text"])
test_labels  = list(cleaned["test"]["label"])

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=MAX_LENGTH)

train_labels_encoded = [int(y) for y in train_labels]
test_labels_encoded  = [int(y) for y in test_labels]

print(f"Train encodings: {len(train_encodings['input_ids'])} samples")
print(f"Test  encodings: {len(test_encodings['input_ids'])}  samples")
print(f"\nSample token preview:")
print(' '.join(train_encodings[1].tokens[:20]))

Train encodings: 16000 samples
Test  encodings: 2000  samples

Sample token preview:
[CLS] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and


In [51]:
# Sanity check — print 5 samples with their decoded labels
for i in range(5):
    tokens = ' '.join(train_encodings[i].tokens[:15])
    label  = id2label[train_labels_encoded[i]]
    print(f"  [{label:8s}]  {tokens}")

  [sadness ]  [CLS] i didn ##t feel humiliated [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
  [sadness ]  [CLS] i can go from feeling so hopeless to so damned hopeful just from being
  [anger   ]  [CLS] im grabbing a minute to post i feel greedy wrong [SEP] [PAD] [PAD] [PAD]
  [love    ]  [CLS] i am ever feeling nos ##tal ##gic about the fireplace i will know that
  [anger   ]  [CLS] i am feeling gr ##ou ##chy [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [52]:
# What BERT actually USES (numerical IDs)
print("\nInput IDs (numbers):")
print(train_encodings[1].ids[:20])

# Attention mask (1 = real token, 0 = padding)
print("\nAttention mask:")
print(train_encodings[1].attention_mask[:20])


Input IDs (numbers):
[101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [53]:
import torch

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MyDataset(train_encodings, train_labels_encoded)
test_dataset  = MyDataset(test_encodings,  test_labels_encoded)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test  dataset: {len(test_dataset)}  samples")
print(f"\nSample item keys: {list(train_dataset[0].keys())}")

Train dataset: 16000 samples
Test  dataset: 2000  samples

Sample item keys: ['input_ids', 'attention_mask', 'labels']


In [54]:
from transformers import DistilBertForSequenceClassification

device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device_name}")

model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label)
).to(device_name)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded: {model_name}  |  {total_params:.1f}M parameters ")

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased  |  67.0M parameters 


In [56]:
def compute_metrics(pred):
    labels = pred.label_ids          # true labels  → [0, 1, 3, 2, 4 ...]
    preds  = pred.predictions.argmax(-1)  # predicted   → [0, 1, 2, 2, 4 ...]
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

In [57]:
# Training arguments
training_args = TrainingArguments(
    output_dir                  = f"./results-{RUN_VERSION}",
    num_train_epochs            = cfg["epochs"],
    per_device_train_batch_size = cfg["batch_size"],
    per_device_eval_batch_size  = 64,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    report_to                   = "wandb",
    run_name                    = f"distilbert-run-{RUN_VERSION}",
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
)

print("Trainer is ready")

Trainer is ready


In [58]:
# Train the model
print(f"Starting training — version {RUN_VERSION} ...")
print(f"lr={cfg['learning_rate']}  batch={cfg['batch_size']}  epochs={cfg['epochs']}")

trainer.train()

print("\nTraining is complete ")

Starting training — version v1 ...
lr=3e-05  batch=16  epochs=3


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.382066,0.419278,0.925500,0.926837
2,0.221758,0.292591,0.930500,0.931437
3,0.154418,0.315335,0.932500,0.931686


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training is complete 


In [59]:
import json
from sklearn.metrics import classification_report

# Run evaluation
eval_results = trainer.evaluate()
print(eval_results)

# Log final metrics to W&B
wandb.log({
    "final/loss":     eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1":       eval_results["eval_f1"],
})

# Save full classification report
preds  = trainer.predict(test_dataset).predictions.argmax(-1)
labels = [item["labels"].item() for item in test_dataset]
report = classification_report(
    labels, preds,
    target_names=list(id2label.values()),
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

# Upload to W&B as a versioned artifact
artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
wandb.log_artifact(artifact)

print("\nEvaluation is complete ")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.154418,0.292591,3,0.930500,0.931437


{'eval_loss': 0.29259127378463745, 'eval_accuracy': 0.9305, 'eval_f1': 0.9314372027665166}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Evaluation is complete 


In [60]:
import json
from sklearn.metrics import classification_report

# Run evaluation
eval_results = trainer.evaluate()
print(eval_results)

# Log final metrics to W&B
wandb.log({
    "final/loss":     eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1":       eval_results["eval_f1"],
})

# Save full classification report
preds  = trainer.predict(test_dataset).predictions.argmax(-1)
labels = [item["labels"].item() for item in test_dataset]
report = classification_report(
    labels, preds,
    target_names=list(id2label.values()),
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

# Upload to W&B as a versioned artifact
artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
wandb.log_artifact(artifact)

print("\nEvaluation complete ")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.154418,0.292591,3,0.930500,0.931437


{'eval_loss': 0.29259127378463745, 'eval_accuracy': 0.9305, 'eval_f1': 0.9314372027665166}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Evaluation complete 


wandb: WARNING Artifact "eval-report" already exists with the same content. No new version will be created.


In [61]:
# # Push model and tokenizer to Hugging Face Hub
# HF_REPO = "https://www.kaggle.com/code/anilg25ait2009/" 
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)

# hf_url = f"https://huggingface.co/Maxii2tj/emotion-distilbert"

# print(f"Model pushed → {hf_url} ")

# # Record in W&B run summary
# wandb.run.summary["huggingface_model"] = hf_url
# wandb.run.summary["final_accuracy"]    = eval_results["eval_accuracy"]
# wandb.run.summary["final_f1"]          = eval_results["eval_f1"]

# wandb.finish()
# print(f"\nRun {RUN_VERSION} complete!")
# print(f"W&B  → https://wandb.ai/maxi-i2tj-na/mlops-group-project")
# print(f"HF   → {hf_url}")

# https://github.com/aniliter-cloud/ml-ops-group-project

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'https://www.kaggle.com/code/anilg25ait2009/mlops-assignment-2-fine-tuning-classification'. Use `repo_type` argument if needed.